# 07 - Multi-Seed Final Evaluation

Aggregate the completed LightGCN, EmerG, DGD, and selected DGD-ablation result
bundles across seeds. This notebook does not retrain models; notebooks 03-06
own training and evaluation. Notebook 07 verifies those bundles, computes
seed-level summaries, flags missing seed coverage, and publishes the final
MovieLens-1M evaluation bundle for tables and figures.

## Notebook Linkage and Work Plan

**Inputs:** notebook-02 protocol identity plus result manifests from notebooks
03-06. Each model bundle already contains validation-selected thresholds and
held-out evaluation metrics. The DGD-ablation bundle also records which variant
was selected using validation only.

**What this notebook does:**
1. Discover local/Kaggle result manifests and verify required artifact hashes.
2. Build a run registry keyed by model family, model label, seed, and bundle ID.
3. Load phase-level evaluation metrics and validation thresholds from every run.
4. Aggregate mean, standard deviation, and standard error by model and phase.
5. Export run registry, per-run metrics, aggregate metrics, prediction-artifact
   registry, reproducibility audit, and a manifest for notebook 08.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 07 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for evaluation notebooks."
    )

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
OUTPUT_ROOT = ARTIFACT_ROOT / "evaluations" / "ml-1m" / "multiseed-v1"
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
EXPECTED_PROTOCOL_SCHEMA = "ml1m-coldstart-v1"
TARGET_SEEDS = [
    int(seed)
    for seed in os.environ.get("COLDSTART_MULTI_SEEDS", "2025,7788,9999,3407,4517").split(",")
    if seed.strip()
]
TARGET_SEED_SET = set(TARGET_SEEDS)
PHASE_ORDER = ["Cold", "Warm A", "Warm B", "Warm C"]
EXPECTED_FAMILIES = {
    "lightgcn": {"schema": "lightgcn-baseline-v1", "label": "LightGCN"},
    "emerg": {"schema": "emerg-baseline-v1", "label": "EmerG"},
    "dgd": {"schema": "dgd-model-v1", "label": "DGD"},
    "dgd_ablation": {"schema": "dgd-ablation-v1", "label": "DGD-Ablation"},
}
SCHEMA_TO_FAMILY = {value["schema"]: key for key, value in EXPECTED_FAMILIES.items()}
RUN_CONFIG = {
    "schema_version": "multiseed-evaluation-v1",
    "target_seeds": TARGET_SEEDS,
    "phase_order": PHASE_ORDER,
    "expected_families": EXPECTED_FAMILIES,
    "selection_policy": "aggregate verified result bundles; do not tune on evaluation metrics",
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "artifact_root": str(ARTIFACT_ROOT),
            "output_root": str(OUTPUT_ROOT),
            "target_seeds": TARGET_SEEDS,
            "run_config_sha256": RUN_CONFIG_SHA256,
        }
    ]
)

,execution_context,python,artifact_root,output_root,target_seeds,run_config_sha256
0,kaggle,3.12.13,/kaggle/working/artifacts,/kaggle/working/artifacts/evaluations/ml-1m/mu...,"[2025, 7788, 9999, 3407, 4517]",34dd820eba950c6d1a503a15e02e49436d4f75d86ee760...


In [2]:
def manifest_schema(manifest: dict[str, Any]) -> str | None:
    return manifest.get("model_schema_version") or manifest.get("ablation_schema_version")


def manifest_status(manifest: dict[str, Any]) -> str | None:
    return manifest.get("model_status") or manifest.get("ablation_status")


def candidate_manifest_paths() -> list[Path]:
    candidates: list[Path] = []
    roots = [PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT]
    if INPUT_ROOT.is_dir():
        roots.append(INPUT_ROOT)
    for root in roots:
        if root.is_dir():
            candidates.extend(root.rglob("manifest.json"))
    unique: list[Path] = []
    seen: set[str] = set()
    for path in sorted(candidates):
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path.resolve())
    return unique


def artifact_root_for(pointer: Path, manifest: dict[str, Any]) -> Path:
    relative_tests = [manifest.get("bundle_manifest")]
    relative_tests.extend(artifact.get("path") for artifact in manifest.get("artifacts", {}).values())
    relative_tests = [str(item) for item in relative_tests if item]
    candidates = [ARTIFACT_ROOT, PROJECT_ROOT / ".notebook" / "artifacts", *pointer.parents]
    for root in candidates:
        try:
            if all((root / relative_path).is_file() for relative_path in relative_tests[:3]):
                return root.resolve()
        except OSError:
            continue
    raise FileNotFoundError(f"Cannot resolve artifact root for {pointer}")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def read_csv_artifact(root: Path, manifest: dict[str, Any], name: str) -> pd.DataFrame:
    artifact = manifest["artifacts"][name]
    path = resolve_inside(root, artifact["path"])
    if sha256_file(path) != artifact["sha256"]:
        raise ValueError(f"Hash mismatch for artifact {name}: {path}")
    return pd.read_csv(path)


def artifact_path_if_valid(root: Path, manifest: dict[str, Any], name: str) -> tuple[str, str] | tuple[None, None]:
    artifact = manifest.get("artifacts", {}).get(name)
    if not artifact:
        return None, None
    path = resolve_inside(root, artifact["path"])
    if not path.is_file() or sha256_file(path) != artifact["sha256"]:
        raise ValueError(f"Hash mismatch for artifact {name}: {path}")
    return str(path), artifact["sha256"]


DISCOVERED_RUNS: list[dict[str, Any]] = []
DISCOVERY_ERRORS: list[str] = []
seen_bundles: set[tuple[str, str]] = set()
for pointer in candidate_manifest_paths():
    try:
        manifest = load_json(pointer)
        schema = manifest_schema(manifest)
        if schema not in SCHEMA_TO_FAMILY:
            continue
        if manifest_status(manifest) != "PASS":
            continue
        bundle_id = str(manifest.get("bundle_id"))
        key = (schema, bundle_id)
        if key in seen_bundles:
            continue
        root = artifact_root_for(pointer, manifest)
        bundle_manifest_path = resolve_inside(root, manifest["bundle_manifest"])
        if bundle_manifest_path.read_bytes() != pointer.read_bytes() and pointer.name == "manifest.json":
            pointer_text = json.dumps(manifest, sort_keys=True)
            bundle_text = json.dumps(load_json(bundle_manifest_path), sort_keys=True)
            if pointer_text != bundle_text:
                raise ValueError("Pointer/generation manifest mismatch")
        family = SCHEMA_TO_FAMILY[schema]
        seed = int(manifest.get("run_config", {}).get("seed", -1))
        if family == "dgd_ablation":
            selected_variant = str(manifest.get("summary", {}).get("advance_to_multiseed", ""))
            model_label = f"DGD-Ablation:{selected_variant}"
        else:
            selected_variant = ""
            model_label = str(manifest.get("model_name", EXPECTED_FAMILIES[family]["label"]))
        DISCOVERED_RUNS.append(
            {
                "family": family,
                "model_label": model_label,
                "selected_variant": selected_variant,
                "schema_version": schema,
                "seed": seed,
                "bundle_id": bundle_id,
                "created_at_utc": manifest.get("created_at_utc", ""),
                "protocol_schema_version": manifest.get("upstream_protocol", {}).get("schema_version", ""),
                "protocol_pointer_sha256": manifest.get("upstream_protocol", {}).get("pointer_sha256", ""),
                "manifest_path": str(pointer),
                "artifact_root": str(root),
                "manifest": manifest,
            }
        )
        seen_bundles.add(key)
    except Exception as error:
        DISCOVERY_ERRORS.append(f"{pointer}: {error}")

if not DISCOVERED_RUNS:
    raise RuntimeError("No verified model result manifests found for notebooks 03-06")

RUN_REGISTRY_RAW = pd.DataFrame([{key: value for key, value in row.items() if key != "manifest"} for row in DISCOVERED_RUNS])
RUN_REGISTRY_RAW = RUN_REGISTRY_RAW.sort_values(["family", "seed", "created_at_utc", "bundle_id"]).reset_index(drop=True)
RUN_REGISTRY = RUN_REGISTRY_RAW.drop_duplicates(["family", "model_label", "seed"], keep="last").reset_index(drop=True)
ACTIVE_KEYS = set(zip(RUN_REGISTRY["family"], RUN_REGISTRY["model_label"], RUN_REGISTRY["seed"], RUN_REGISTRY["bundle_id"]))
ACTIVE_RUNS = [
    row
    for row in DISCOVERED_RUNS
    if (row["family"], row["model_label"], row["seed"], row["bundle_id"]) in ACTIVE_KEYS
]

display(RUN_REGISTRY[["family", "model_label", "seed", "bundle_id", "created_at_utc"]])
if DISCOVERY_ERRORS:
    show_records([{"discovery_warning": error} for error in DISCOVERY_ERRORS[:10]])

,family,model_label,seed,bundle_id,created_at_utc
0,dgd,DGD,2025,20260716T030807-4a25ee4b64a5,2026-07-16T03:08:09.678427+00:00
1,dgd_ablation,DGD-Ablation:full_dgd,2025,20260716T040921-40a1cb31a2e0,2026-07-16T04:09:25.431835+00:00
2,emerg,EmerG,2025,20260716T021911-455f535b027d,2026-07-16T02:19:14.430500+00:00
3,lightgcn,LightGCN,2025,20260716T013724-7f5d845a2436,2026-07-16T01:37:26.930554+00:00


In [3]:
metric_parts: list[pd.DataFrame] = []
threshold_parts: list[pd.DataFrame] = []
prediction_registry_rows: list[dict[str, Any]] = []
artifact_audit_rows: list[dict[str, Any]] = []

for run in ACTIVE_RUNS:
    manifest = run["manifest"]
    root = Path(run["artifact_root"])
    metrics = read_csv_artifact(root, manifest, "evaluation_metrics")
    thresholds = read_csv_artifact(root, manifest, "validation_thresholds")
    prediction_path, prediction_sha = artifact_path_if_valid(root, manifest, "evaluation_predictions")

    if run["family"] == "dgd_ablation":
        selected = run["selected_variant"]
        if not selected:
            raise ValueError(f"Ablation bundle {run['bundle_id']} does not declare advance_to_multiseed")
        metrics = metrics[metrics["variant"].astype(str).eq(selected)].copy()
        thresholds = thresholds[thresholds["variant"].astype(str).eq(selected)].copy()
        if metrics.empty or thresholds.empty:
            raise ValueError(f"Ablation bundle {run['bundle_id']} has no rows for selected variant {selected!r}")
    else:
        if "variant" not in metrics.columns:
            metrics["variant"] = ""
        if "variant" not in thresholds.columns:
            thresholds["variant"] = ""

    for table in (metrics, thresholds):
        table.insert(0, "family", run["family"])
        table.insert(1, "model_label", run["model_label"])
        table.insert(2, "seed", run["seed"])
        table.insert(3, "bundle_id", run["bundle_id"])
        table.insert(4, "schema_version", run["schema_version"])
        table.insert(5, "protocol_pointer_sha256", run["protocol_pointer_sha256"])

    metric_parts.append(metrics)
    threshold_parts.append(thresholds)
    prediction_registry_rows.append(
        {
            "family": run["family"],
            "model_label": run["model_label"],
            "seed": run["seed"],
            "bundle_id": run["bundle_id"],
            "evaluation_predictions_path": prediction_path,
            "evaluation_predictions_sha256": prediction_sha,
        }
    )
    for name in ("evaluation_metrics", "validation_thresholds", "evaluation_predictions"):
        artifact = manifest.get("artifacts", {}).get(name)
        artifact_audit_rows.append(
            {
                "family": run["family"],
                "model_label": run["model_label"],
                "seed": run["seed"],
                "bundle_id": run["bundle_id"],
                "artifact": name,
                "present": artifact is not None,
                "sha256": artifact.get("sha256") if artifact else "",
            }
        )

PER_RUN_METRICS = pd.concat(metric_parts, ignore_index=True)
VALIDATION_THRESHOLDS = pd.concat(threshold_parts, ignore_index=True)
PREDICTION_ARTIFACT_REGISTRY = pd.DataFrame(prediction_registry_rows)
ARTIFACT_AUDIT = pd.DataFrame(artifact_audit_rows)

if not set(PHASE_ORDER) <= set(PER_RUN_METRICS["phase"].astype(str)):
    raise RuntimeError("Per-run metrics do not cover all required phases")

display(PER_RUN_METRICS[["model_label", "seed", "phase", "f1", "roc_auc"]].sort_values(["model_label", "seed", "phase"]))

,model_label,seed,phase,f1,roc_auc
8,DGD,2025,Cold,0.612464,0.732979
9,DGD,2025,Warm A,0.670894,0.782171
10,DGD,2025,Warm B,0.688299,0.796279
11,DGD,2025,Warm C,0.690961,0.801468
12,DGD-Ablation:full_dgd,2025,Cold,0.615625,0.732799
13,DGD-Ablation:full_dgd,2025,Warm A,0.651598,0.766125
14,DGD-Ablation:full_dgd,2025,Warm B,0.672968,0.784364
15,DGD-Ablation:full_dgd,2025,Warm C,0.685226,0.795544
4,EmerG,2025,Cold,0.612366,0.741064
5,EmerG,2025,Warm A,0.677046,0.794260


In [4]:
METRIC_COLUMNS = ["accuracy", "precision", "recall", "f1", "roc_auc", "predicted_positive_rate"]
available_metric_columns = [column for column in METRIC_COLUMNS if column in PER_RUN_METRICS.columns]

aggregate_rows: list[dict[str, Any]] = []
for (family, model_label, phase), group in PER_RUN_METRICS.groupby(["family", "model_label", "phase"], observed=True):
    observed_seeds = set(group["seed"].astype(int))
    row: dict[str, Any] = {
        "family": family,
        "model_label": model_label,
        "phase": phase,
        "seed_count": int(group["seed"].nunique()),
        "target_seed_count_observed": len(observed_seeds & TARGET_SEED_SET),
        "target_seed_count": len(TARGET_SEEDS),
        "target_seed_count_met": bool(TARGET_SEED_SET <= observed_seeds),
    }
    for metric in available_metric_columns:
        values = pd.to_numeric(group[metric], errors="coerce")
        row[f"{metric}_mean"] = float(values.mean())
        row[f"{metric}_std"] = float(values.std(ddof=1)) if values.count() > 1 else np.nan
        row[f"{metric}_sem"] = float(values.sem(ddof=1)) if values.count() > 1 else np.nan
    aggregate_rows.append(row)

AGGREGATE_METRICS = pd.DataFrame(aggregate_rows)
phase_rank_rows: list[dict[str, Any]] = []
for phase, group in AGGREGATE_METRICS.groupby("phase", observed=True):
    ranked = group.sort_values(["f1_mean", "roc_auc_mean"], ascending=False).reset_index(drop=True)
    for rank, record in enumerate(ranked.to_dict(orient="records"), start=1):
        phase_rank_rows.append({"phase": phase, "rank": rank, **record})
PHASE_RANKING = pd.DataFrame(phase_rank_rows)
OVERALL_RANKING = (
    AGGREGATE_METRICS.groupby(["family", "model_label"], observed=True)
    .agg(
        phases=("phase", "nunique"),
        seed_count_min=("seed_count", "min"),
        f1_mean=("f1_mean", "mean"),
        roc_auc_mean=("roc_auc_mean", "mean"),
    )
    .reset_index()
    .sort_values(["f1_mean", "roc_auc_mean"], ascending=False)
    .reset_index(drop=True)
)
OVERALL_RANKING.insert(0, "rank", np.arange(1, len(OVERALL_RANKING) + 1))

display(AGGREGATE_METRICS.sort_values(["model_label", "phase"]))
display(OVERALL_RANKING)

,family,model_label,phase,seed_count,target_seed_count_observed,target_seed_count,target_seed_count_met,accuracy_mean,accuracy_std,accuracy_sem,...,recall_sem,f1_mean,f1_std,f1_sem,roc_auc_mean,roc_auc_std,roc_auc_sem,predicted_positive_rate_mean,predicted_positive_rate_std,predicted_positive_rate_sem
0,dgd,DGD,Cold,1,1,5,False,0.500651,NaN,NaN,...,NaN,0.612464,NaN,NaN,0.732979,NaN,NaN,0.880300,NaN,NaN
1,dgd,DGD,Warm A,1,1,5,False,0.656482,NaN,NaN,...,NaN,0.670894,NaN,NaN,0.782171,NaN,NaN,0.635568,NaN,NaN
2,dgd,DGD,Warm B,1,1,5,False,0.704320,NaN,NaN,...,NaN,0.688299,NaN,NaN,0.796279,NaN,NaN,0.540378,NaN,NaN
3,dgd,DGD,Warm C,1,1,5,False,0.714812,NaN,NaN,...,NaN,0.690961,NaN,NaN,0.801468,NaN,NaN,0.514600,NaN,NaN
4,dgd_ablation,DGD-Ablation:full_dgd,Cold,1,1,5,False,0.511994,NaN,NaN,...,NaN,0.615625,NaN,NaN,0.732799,NaN,NaN,0.861384,NaN,NaN
5,dgd_ablation,DGD-Ablation:full_dgd,Warm A,1,1,5,False,0.614706,NaN,NaN,...,NaN,0.651598,NaN,NaN,0.766125,NaN,NaN,0.697667,NaN,NaN
6,dgd_ablation,DGD-Ablation:full_dgd,Warm B,1,1,5,False,0.661015,NaN,NaN,...,NaN,0.672968,NaN,NaN,0.784364,NaN,NaN,0.628324,NaN,NaN
7,dgd_ablation,DGD-Ablation:full_dgd,Warm C,1,1,5,False,0.699491,NaN,NaN,...,NaN,0.685226,NaN,NaN,0.795544,NaN,NaN,0.546457,NaN,NaN
8,emerg,EmerG,Cold,1,1,5,False,0.499366,NaN,NaN,...,NaN,0.612366,NaN,NaN,0.741064,NaN,NaN,0.883288,NaN,NaN
9,emerg,EmerG,Warm A,1,1,5,False,0.655891,NaN,NaN,...,NaN,0.677046,NaN,NaN,0.794260,NaN,NaN,0.657281,NaN,NaN


,rank,family,model_label,phases,seed_count_min,f1_mean,roc_auc_mean
0,1,emerg,EmerG,4,1,0.667853,0.784798
1,2,dgd,DGD,4,1,0.665655,0.778224
2,3,dgd_ablation,DGD-Ablation:full_dgd,4,1,0.656354,0.769708
3,4,lightgcn,LightGCN,4,1,0.579735,0.503708


In [5]:
AUDIT_ROWS: list[dict[str, Any]] = []


def audit(name: str, condition: bool, observed: Any, expected: Any, severity: str = "ERROR") -> None:
    AUDIT_ROWS.append(
        {
            "check": name,
            "status": "PASS" if condition else severity,
            "observed": observed,
            "expected": expected,
            "severity": severity,
        }
    )


present_families = set(RUN_REGISTRY["family"])
protocol_schemas = sorted(set(RUN_REGISTRY["protocol_schema_version"].astype(str)))
protocol_hashes = sorted(set(RUN_REGISTRY["protocol_pointer_sha256"].astype(str)))
ablation_labels = sorted(RUN_REGISTRY.loc[RUN_REGISTRY["family"].eq("dgd_ablation"), "model_label"].unique())
seed_coverage_rows: list[dict[str, Any]] = []
for (family, model_label), group in RUN_REGISTRY.groupby(["family", "model_label"], observed=True):
    observed_seeds = set(group["seed"].astype(int))
    for seed in TARGET_SEEDS:
        seed_coverage_rows.append(
            {
                "family": family,
                "model_label": model_label,
                "seed": seed,
                "expected_seed": True,
                "present": seed in observed_seeds,
            }
        )
    for seed in sorted(observed_seeds - TARGET_SEED_SET):
        seed_coverage_rows.append(
            {
                "family": family,
                "model_label": model_label,
                "seed": seed,
                "expected_seed": False,
                "present": True,
            }
        )
SEED_COVERAGE = pd.DataFrame(seed_coverage_rows)
missing_seed_map = (
    SEED_COVERAGE.loc[SEED_COVERAGE["expected_seed"] & ~SEED_COVERAGE["present"]]
    .groupby("model_label", observed=True)["seed"]
    .apply(lambda values: sorted(int(value) for value in values))
    .to_dict()
)
target_seed_complete = not missing_seed_map
threshold_pairs = set(zip(VALIDATION_THRESHOLDS["model_label"], VALIDATION_THRESHOLDS["seed"], VALIDATION_THRESHOLDS["phase"]))
metric_pairs = set(zip(PER_RUN_METRICS["model_label"], PER_RUN_METRICS["seed"], PER_RUN_METRICS["phase"]))

audit("required model families present", set(EXPECTED_FAMILIES) <= present_families, sorted(present_families), sorted(EXPECTED_FAMILIES))
audit("protocol schema", protocol_schemas == [EXPECTED_PROTOCOL_SCHEMA], protocol_schemas, [EXPECTED_PROTOCOL_SCHEMA])
audit("single protocol identity", len(protocol_hashes) == 1 and bool(protocol_hashes[0]), protocol_hashes, "one nonempty protocol pointer sha256")
audit("phase coverage", set(PHASE_ORDER) <= set(PER_RUN_METRICS["phase"].astype(str)), sorted(PER_RUN_METRICS["phase"].unique()), PHASE_ORDER)
audit("threshold coverage", metric_pairs <= threshold_pairs, len(threshold_pairs), f">= {len(metric_pairs)}")
audit("prediction artifacts verified", PREDICTION_ARTIFACT_REGISTRY["evaluation_predictions_path"].notna().all(), int(PREDICTION_ARTIFACT_REGISTRY["evaluation_predictions_path"].notna().sum()), len(PREDICTION_ARTIFACT_REGISTRY))
audit("selected ablation stable", len(ablation_labels) <= 1, ablation_labels, "one selected ablation label", severity="WARN")
audit("target five seeds complete", target_seed_complete, missing_seed_map, "no missing target seeds", severity="WARN")

REPRO_AUDIT = pd.DataFrame(AUDIT_ROWS)
EVALUATION_PASS = not REPRO_AUDIT["status"].eq("ERROR").any()
display(REPRO_AUDIT)
if not EVALUATION_PASS:
    raise RuntimeError("Blocking multi-seed evaluation audit checks failed")

,check,status,observed,expected,severity
0,required model families present,PASS,"[dgd, dgd_ablation, emerg, lightgcn]","[dgd, dgd_ablation, emerg, lightgcn]",ERROR
1,protocol schema,PASS,[ml1m-coldstart-v1],[ml1m-coldstart-v1],ERROR
2,single protocol identity,PASS,[f6461f73814475752e5db6bf6b5ae373ca92e3c049aff...,one nonempty protocol pointer sha256,ERROR
3,phase coverage,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]",ERROR
4,threshold coverage,PASS,16,>= 16,ERROR
5,prediction artifacts verified,PASS,4,4,ERROR
6,selected ablation stable,PASS,[DGD-Ablation:full_dgd],one selected ablation label,WARN
7,target five seeds complete,WARN,"{'DGD': [3407, 4517, 7788, 9999], 'DGD-Ablatio...",no missing target seeds,WARN


In [6]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


bundle_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
staging_root = OUTPUT_ROOT / f".staging-{bundle_id}"
generation_root = OUTPUT_ROOT / "generations" / bundle_id
pointer_path = OUTPUT_ROOT / "manifest.json"
staging_root.mkdir(parents=True, exist_ok=False)

OUTPUT_TABLES = {
    "run_registry": RUN_REGISTRY.drop(columns=["artifact_root"], errors="ignore"),
    "per_run_metrics": PER_RUN_METRICS,
    "validation_thresholds": VALIDATION_THRESHOLDS,
    "aggregate_metrics": AGGREGATE_METRICS,
    "phase_ranking": PHASE_RANKING,
    "overall_ranking": OVERALL_RANKING,
    "seed_coverage": SEED_COVERAGE,
    "prediction_artifact_registry": PREDICTION_ARTIFACT_REGISTRY,
    "artifact_audit": ARTIFACT_AUDIT,
    "reproducibility_audit": REPRO_AUDIT,
}
OUTPUT_ARTIFACTS: dict[str, Any] = {}

try:
    for name, table in OUTPUT_TABLES.items():
        staging_path = staging_root / f"{name}.csv"
        published_path = generation_root / f"{name}.csv"
        write_csv(staging_path, table)
        OUTPUT_ARTIFACTS[name] = {"path": relative_output(published_path), "sha256": sha256_file(staging_path), "rows": len(table)}

    manifest = {
        "evaluation_schema_version": RUN_CONFIG["schema_version"],
        "evaluation_status": "PASS",
        "coverage_status": "COMPLETE" if target_seed_complete else "PARTIAL",
        "bundle_id": bundle_id,
        "bundle_manifest": relative_output(generation_root / "manifest.json"),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_config": RUN_CONFIG,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "upstream_protocol_pointer_sha256": protocol_hashes[0] if protocol_hashes else "",
        "summary": {
            "models": int(RUN_REGISTRY["model_label"].nunique()),
            "runs": int(len(RUN_REGISTRY)),
            "target_seed_count": len(TARGET_SEEDS),
            "target_seed_complete": bool(target_seed_complete),
            "missing_target_seeds": missing_seed_map,
            "best_model_by_mean_f1": str(OVERALL_RANKING.iloc[0]["model_label"]),
            "best_mean_f1": float(OVERALL_RANKING.iloc[0]["f1_mean"]),
            "best_mean_auc": float(OVERALL_RANKING.iloc[0]["roc_auc_mean"]),
        },
        "checks": REPRO_AUDIT.to_dict(orient="records"),
        "artifacts": OUTPUT_ARTIFACTS,
        "output_schemas": {name: csv_schema(table) for name, table in OUTPUT_TABLES.items()},
        "training_contract": {
            "source": "verified notebooks 03-06 result bundles",
            "thresholds": "loaded from validation-selected thresholds in each source bundle",
            "evaluation": "held-out evaluation metrics only; no threshold or model selection performed here",
        },
    }

    manifest_text = json.dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n"
    write_text(staging_root / "manifest.json", manifest_text)
    generation_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(generation_root)
    generation_hashes_match = all(
        sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
        for artifact in OUTPUT_ARTIFACTS.values()
    )
    if not generation_hashes_match or (generation_root / "manifest.json").read_text(encoding="utf-8") != manifest_text:
        raise RuntimeError("Published multi-seed generation failed pre-pointer verification")
    write_text(pointer_path, manifest_text)
except Exception:
    if staging_root.exists():
        shutil.rmtree(staging_root, ignore_errors=True)
    raise

MULTISEED_MANIFEST = manifest
MULTISEED_POINTER = pointer_path
show_records(
    [
        {
            "evaluation_status": "PASS",
            "coverage_status": manifest["coverage_status"],
            "bundle_id": bundle_id,
            "manifest": str(pointer_path),
            "artifacts": len(OUTPUT_ARTIFACTS),
        }
    ]
)

,evaluation_status,coverage_status,bundle_id,manifest,artifacts
0,PASS,PARTIAL,20260716T043713-d8f1f902ce96,/kaggle/working/artifacts/evaluations/ml-1m/mu...,10


In [7]:
FINAL_CHECKS: list[dict[str, Any]] = list(REPRO_AUDIT.to_dict(orient="records"))


def final_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    FINAL_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "ERROR", "observed": observed, "expected": expected, "severity": "ERROR"}
    )


manifest_pointer_matches = MULTISEED_POINTER.read_text(encoding="utf-8") == (generation_root / "manifest.json").read_text(encoding="utf-8")
artifact_hashes_match = all(
    sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
    for artifact in MULTISEED_MANIFEST["artifacts"].values()
)
final_check("manifest pointer matches bundle", manifest_pointer_matches, manifest_pointer_matches, True)
final_check("artifact hashes verify", artifact_hashes_match, artifact_hashes_match, True)
final_check("aggregate metrics exported", "aggregate_metrics" in MULTISEED_MANIFEST["artifacts"], True, True)
final_check("prediction registry exported", "prediction_artifact_registry" in MULTISEED_MANIFEST["artifacts"], True, True)

FINAL_AUDIT = pd.DataFrame(FINAL_CHECKS)
FINAL_PASS = not FINAL_AUDIT["status"].eq("ERROR").any()
display(FINAL_AUDIT)
display(Markdown("### Notebook 07 multi-seed evaluation: " + ("READY" if FINAL_PASS else "BLOCKED")))

if not FINAL_PASS:
    raise RuntimeError("Final multi-seed export checks failed")

display(
    Markdown(
        "**Next notebook:** notebook 08 should read this multiseed manifest and build the final tables/figures. "
        "If coverage is PARTIAL, rerun notebooks 03-06 with the missing seeds and rerun notebook 07."
    )
)

,check,status,observed,expected,severity
0,required model families present,PASS,"[dgd, dgd_ablation, emerg, lightgcn]","[dgd, dgd_ablation, emerg, lightgcn]",ERROR
1,protocol schema,PASS,[ml1m-coldstart-v1],[ml1m-coldstart-v1],ERROR
2,single protocol identity,PASS,[f6461f73814475752e5db6bf6b5ae373ca92e3c049aff...,one nonempty protocol pointer sha256,ERROR
3,phase coverage,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]",ERROR
4,threshold coverage,PASS,16,>= 16,ERROR
5,prediction artifacts verified,PASS,4,4,ERROR
6,selected ablation stable,PASS,[DGD-Ablation:full_dgd],one selected ablation label,WARN
7,target five seeds complete,WARN,"{'DGD': [3407, 4517, 7788, 9999], 'DGD-Ablatio...",no missing target seeds,WARN
8,manifest pointer matches bundle,PASS,True,True,ERROR
9,artifact hashes verify,PASS,True,True,ERROR


### Notebook 07 multi-seed evaluation: READY

**Next notebook:** notebook 08 should read this multiseed manifest and build the final tables/figures. If coverage is PARTIAL, rerun notebooks 03-06 with the missing seeds and rerun notebook 07.